In [ ]:
from google.colab import files

uploaded = files.upload()   # select all 115 files at once in the dialog box

Saving 17 apr.xls to 17 apr.xls
Saving 17 aug.xls to 17 aug.xls
Saving 17 dec.xls to 17 dec.xls
Saving 17 feb.xls to 17 feb.xls
Saving 17 jan.xls to 17 jan.xls
Saving 17 july.xls to 17 july.xls
Saving 17 june.xls to 17 june.xls
Saving 17 mar.xls to 17 mar.xls
Saving 17 may.xls to 17 may.xls
Saving 17 nov.xls to 17 nov.xls
Saving 17 oct.xls to 17 oct.xls
Saving 17 sept.xls to 17 sept.xls
Saving 18 apr.xls to 18 apr.xls
Saving 18 aug.xls to 18 aug.xls
Saving 18 dec.xls to 18 dec.xls
Saving 18 feb.xls to 18 feb.xls
Saving 18 jan.xls to 18 jan.xls
Saving 18 july.xls to 18 july.xls
Saving 18 june.xls to 18 june.xls
Saving 18 mar.xls to 18 mar.xls
Saving 18 may.xls to 18 may.xls
Saving 18 nov.xls to 18 nov.xls
Saving 18 oct.xls to 18 oct.xls
Saving 18 sept.xls to 18 sept.xls
Saving 19 apr.xls to 19 apr.xls
Saving 19 aug.xls to 19 aug.xls
Saving 19 dec.xls to 19 dec.xls
Saving 19 feb.xls to 19 feb.xls
Saving 19 jan.xls to 19 jan.xls
Saving 19 july.xls to 19 july.xls
Saving 19 june.xls to 19 j

In [ ]:
import glob
import os
import pandas as pd


def parse_nbp_file(file_path):
    """Parses a single Life Insurance Council HTML-Excel file."""
    dfs = pd.read_html(file_path)
    df_detailed = dfs[2]  # Table 2 contains detailed company-wise breakdown

    # Extract Report Month string from header
    header_text = str(df_detailed.iloc[2, 2])
    month_str = header_text.replace("FOR THE MONTH ", "").strip()

    current_company = None
    records = []

    valid_products = [
        "Individual Single Premium",
        "Individual Non Single Premium",
        "Group Single Premium",
        "Group Non Single Premium",
        "Group Yearly Renewable Premium",
        "Total",
    ]

    for idx in range(3, len(df_detailed)):
        s_no = df_detailed.iloc[idx, 0]
        particular = df_detailed.iloc[idx, 1]

        if pd.isna(particular):
            continue

        # Detect Company Name row (starts with serial number)
        if pd.notna(s_no) and str(s_no).isdigit():
            current_company = str(particular).strip()
            continue

        # Extract product line data
        if current_company and str(particular).strip() in valid_products:
            records.append({
                "Report_Month": month_str,
                "Company": current_company,
                "Product_Category": str(particular).strip(),
                "Premium_Current_Month": pd.to_numeric(
                    df_detailed.iloc[idx, 2], errors="coerce"
                ),
                "Premium_YTD": pd.to_numeric(
                    df_detailed.iloc[idx, 3], errors="coerce"
                ),
                "Policies_Current_Month": pd.to_numeric(
                    df_detailed.iloc[idx, 7], errors="coerce"
                ),
                "Policies_YTD": pd.to_numeric(
                    df_detailed.iloc[idx, 8], errors="coerce"
                ),
            })

    return pd.DataFrame(records)


def main():
    raw_files = glob.glob("*.xls") + glob.glob("*.xlsx")
    print(f"Found {len(raw_files)} files")

    all_dfs = []
    for f in raw_files:
        try:
            df = parse_nbp_file(f)
            all_dfs.append(df)
            print(f"Successfully processed: {f}")
        except Exception as e:
            print(f"Failed to process {f}: {e}")

    if all_dfs:
        master_df = pd.concat(all_dfs, ignore_index=True)

        # Convert date to standard datetime format
        master_df["Date"] = pd.to_datetime(
            master_df["Report_Month"], format="%B-%Y", errors="coerce"
        )
        master_df = master_df.sort_values(
            by=["Date", "Company", "Product_Category"]
        )

        # Ensure output folder exists
        os.makedirs("data/processed", exist_ok=True)

        output_path = "data/processed/master_sales_data.csv"
        master_df.to_csv(output_path, index=False)
        print(f"\nMaster dataset saved to {output_path}!")
        print(f"Total rows: {len(master_df)}")
    else:
        print("No files were successfully processed.")


if __name__ == "__main__":
    main()

Found 115 files
Successfully processed: 19 sept.xls
Successfully processed: 20 jan.xls
Successfully processed: 18 dec.xls
Successfully processed: DetailedReportofNB (2023_may).xls
Successfully processed: 19 apr.xls
Successfully processed: 17 sept.xls
Successfully processed: DetailedReportofNB (2025_may).xls
Successfully processed: DetailedReportofNB (2025_october).xls
Successfully processed: 18 sept.xls
Successfully processed: DetailedReportofNB (2021_february).xls
Successfully processed: 18 june.xls
Successfully processed: DetailedReportofNB (2025_march).xls
Successfully processed: DetailedReportofNB (2026_may).xls
Successfully processed: 19 jan.xls
Successfully processed: DetailedReportofNB (2024_january).xls
Successfully processed: 17 apr.xls
Successfully processed: 17 feb.xls
Successfully processed: DetailedReportofNB (2023_september).xls
Successfully processed: 18 mar.xls
Successfully processed: 17 aug.xls
Successfully processed: 19 aug.xls
Successfully processed: 19 feb.xls
Succe